# SECTION 0 — Environment Setup

In [ ]:
!pip install librosa soundfile openpyxl lightgbm noisereduce shap

# Imports:

import os
import numpy as np
import pandas as pd

import librosa
import noisereduce as nr

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import KNNImputer

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix,
    roc_auc_score
)

from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

from lightgbm import LGBMClassifier

from sklearn.ensemble import RandomForestClassifier

In [ ]:
import os, glob, joblib
import numpy as np
import pandas as pd
import librosa
import noisereduce as nr

from sklearn.impute import KNNImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from lightgbm import LGBMClassifier
from sklearn.base import clone
from sklearn.metrics import classification_report, confusion_matrix, matthews_corrcoef

# SECTION 1 — Define Dataset Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MODMA_PATH='/content/drive/MyDrive/MODMA_audio'
AUDIO_PATH=MODMA_PATH+'/audio_lanzhou_2015'
LABEL_PATH=MODMA_PATH+'/EEG_128channels_resting_lanzhou_2015/subjects_information_EEG_128channels_resting_lanzhou_2015.xlsx'
BEHAVIOR_PATH='/content/Dataset-Mental-Disorders.csv'

# SECTION 2 — MODMA Label Mapping

Read:
subjects_information_EEG_128channels_resting_lanzhou_2015.xlsx

Create:

In [ ]:
Subject_ID → Depression/Normal

Example

In [ ]:
label_mapping = {

"02010001":1,

"02010004":0

}

# SECTION 3 — MODMA Audio Pipeline
Audio discovery

In [ ]:
import glob


audio_files = glob.glob(
    AUDIO_PATH+"/**/*.wav",
    recursive=True
)

Extract:

In [ ]:
subject_id = file.split("/")[-2]

Match With:

In [ ]:
label_mapping

Output:

audio_path | subject | label

# SECTION 4 — Voice Feature Extraction
Following the paper:

## Audio
 |
## STE segmentation
 |
## Spectral gating
 |
## MFCC
## Pitch
## Jitter
## Shimmer
## HNR

# MFCC

In [ ]:
def mfcc_features(y,sr):

    mfcc = librosa.feature.mfcc(
        y=y,
        sr=sr,
        n_mfcc=13
    )

    return np.concatenate(
        [
        mfcc.mean(axis=1),
        mfcc.std(axis=1)
        ]
    )

# Pitch

In [ ]:
def pitch_feature(y,sr):

    pitches,_ = librosa.piptrack(
        y=y,
        sr=sr
    )

    return np.mean(
        pitches[pitches>0]
    )

# Jitter/Shimmer/HNR

These will be implemented using frame-level F0/amplitude variation.

# SECTION 5 — Behavioral Dataset

Load:

In [ ]:
behavior=pd.read_csv(
    BEHAVIOR_PATH
)

## Convert:
Normal → 0

Depression/Bipolar → 1

In [ ]:
class IRF:
    def __init__(self,n_estimators=100):
        self.models=[]
        self.weights=[]
        self.n_estimators=n_estimators

    def fit(self,X,y):
        for _ in range(self.n_estimators):
            model=RandomForestClassifier(n_estimators=1,max_depth=20)
            model.fit(X,y)
            self.models.append(model)
            self.weights.append(model.score(X,y))

    def predict(self,X):
        votes=[m.predict(X)*w for m,w in zip(self.models,self.weights)]
        return (np.mean(votes,axis=0)>0.5).astype(int)

# SECTION 6 — Behavioral Preprocessing
## KNN Imputation

In [ ]:
imputer=KNNImputer(
n_neighbors=5
)

X_behavior=imputer.fit_transform(
X_behavior
)

## Min-Max normalization

In [ ]:
scaler=MinMaxScaler()

X_behavior=scaler.fit_transform(
X_behavior
)

# SECTION 7 — Statistical Feature Extraction

The paper uses:

### 1.Mean
### 2.Variance
### 3.Skewness

Implementation:

In [ ]:
from scipy.stats import skew


def statistical_features(X):

    return np.column_stack(

    [
    np.mean(X,axis=1),

    np.var(X,axis=1),

    skew(X,axis=1)

    ]

    )

# SECTION 8 — IDTW Implementation ⭐

This will be included.

Improved DTW

In [ ]:
def IDTW(
    sequence1,
    sequence2,
    weights=None
):


    n=len(sequence1)

    m=len(sequence2)


    if weights is None:

        weights=np.ones(n)



    cost=np.zeros(
        (n+1,m+1)
    )


    cost[:]=np.inf

    cost[0,0]=0



    for i in range(1,n+1):

        for j in range(1,m+1):


            distance = (

                weights[i-1]
                *
                abs(
                    sequence1[i-1]
                    -
                    sequence2[j-1]
                )

            )


            cost[i,j]=distance+min(

                cost[i-1,j],

                cost[i,j-1],

                cost[i-1,j-1]

            )


    return cost[n,m]

## Generate IDTW Features

In [ ]:
def IDTW(
    sequence1,
    sequence2,
    weights=None
):


    n=len(sequence1)

    m=len(sequence2)


    if weights is None:

        weights=np.ones(n)



    cost=np.zeros(
        (n+1,m+1)
    )


    cost[:]=np.inf

    cost[0,0]=0



    for i in range(1,n+1):

        for j in range(1,m+1):


            distance = (

                weights[i-1]
                *
                abs(
                    sequence1[i-1]
                    -
                    sequence2[j-1]
                )

            )


            cost[i,j]=distance+min(

                cost[i-1,j],

                cost[i,j-1],

                cost[i-1,j-1]

            )


    return cost[n,m]

### Final behavioral feature matrix:

In [ ]:
X_behavior_final = np.concatenate(

[
statistical_features(X_behavior),

IDTW_features

],

axis=1

)

# SECTION 9 — Paper-Specific MRFE ⭐

Normal RFE:

Train once
Remove features

MRFE:

### Train model


↓


### Calculate feature importance


↓


### Remove weakest features


↓


### Retrain model


↓


### Repeat

Implementation:

In [ ]:
class MRFE:


    def __init__(
        self,
        estimator,
        n_features
    ):

        self.estimator=estimator

        self.n_features=n_features



    def fit_transform(
        self,
        X,
        y
    ):


        features=list(
            range(
            X.shape[1]
            )
        )


        while len(features)>self.n_features:


            self.estimator.fit(
                X[:,features],
                y
            )


            importance=(

            self.estimator
            .feature_importances_

            )


            weakest=np.argmin(
                importance
            )


            del features[weakest]


        self.selected_features=features


        return X[:,features]

# SECTION 10 — Improved Random Forest (IRF) ⭐

Paper-specific concept:

Instead of equal voting:

Tree 1 vote = Tree accuracy weight


Tree 2 vote = Tree accuracy weight

Implementation:

In [ ]:
class ImprovedRandomForest:


    def __init__(self,n_estimators=100):

        self.models=[]

        self.weights=[]

        self.n_estimators=n_estimators



    def fit(self,X,y):


        for i in range(
            self.n_estimators
        ):

            model=RandomForestClassifier(
                n_estimators=1,
                max_depth=20
            )


            model.fit(
                X,
                y
            )


            acc=model.score(
                X,
                y
            )


            self.models.append(model)

            self.weights.append(acc)



    def predict(self,X):

        votes=[]


        for model,weight in zip(
            self.models,
            self.weights
        ):

            votes.append(
                model.predict(X)
                *
                weight
            )


        return (
        np.mean(votes,axis=0)
        >0.5
        ).astype(int)

# SECTION 11 — LightGBM Branch

In [ ]:
LGBM=LGBMClassifier(

learning_rate=0.1,

num_leaves=20,

max_depth=5

)

# SECTION 12 — Hybrid SVM-KNN ⭐

In [ ]:
class HybridSVMKNN:


    def __init__(self):

        self.svm=SVC(
            kernel="rbf",
            C=0.1,
            probability=True
        )

        self.knn=KNeighborsClassifier(
            n_neighbors=5
        )


    def fit(self,X,y):

        self.svm.fit(X,y)

        self.knn.fit(X,y)



    def predict(self,X):

        svm_prob=(
            self.svm
            .predict_proba(X)
        )


        confidence=np.max(
            svm_prob,
            axis=1
        )


        svm_pred=self.svm.predict(X)

        knn_pred=self.knn.predict(X)



        return np.where(

            confidence>0.2,

            svm_pred,

            knn_pred

        )

# SECTION 13 — NeuroVibeNet Fusion

## Final:

## IRF prediction
# +
## LightGBM prediction
# +
## Hybrid SVM-KNN prediction

In [ ]:
final_score=(

0.33*irf_prediction

+

0.33*lgbm_prediction

+

0.34*voice_prediction

)


final_prediction = (
final_score>0.5
).astype(int)

# SECTION 14 — Evaluation

Metrics:

In [ ]:
Accuracy

Precision

Recall

F1-score

MCC

ROC-AUC

Confusion Matrix

## Hybrid SVM-KNN

In [ ]:
class HybridSVMKNN:
    def __init__(self):
        self.svm=SVC(kernel='rbf',C=0.1,probability=True)
        self.knn=KNeighborsClassifier(n_neighbors=5)

    def fit(self,X,y):
        self.svm.fit(X,y)
        self.knn.fit(X,y)

    def predict(self,X):
        conf=np.max(self.svm.predict_proba(X),axis=1)
        return np.where(conf>0.2,self.svm.predict(X),self.knn.predict(X))

## Fusion Model, Evaluation and Saving

In [ ]:
def fusion(irf,lgbm,voice):
    score=0.33*irf+0.33*lgbm+0.34*voice
    return (score>0.5).astype(int)

def evaluate(y_true,y_pred):
    print(classification_report(y_true,y_pred))
    print('MCC:',matthews_corrcoef(y_true,y_pred))
    print(confusion_matrix(y_true,y_pred))

# Example:
# joblib.dump(model,'model.pkl')